# Exercice 2: Classification system with KNN - To Loan or Not To Loan

## Imports

Import some useful libraries

In [48]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split

## a. Getting started

### Data loading

The original dataset comes from the Kaggle's [Loan Prediction](https://www.kaggle.com/ninzaami/loan-predication) problem. The provided dataset has already undergone some processing, such as removing some columns and invalid data. Pandas is used to read the CSV file.

In [49]:
data = pd.read_csv("loandata.csv")

Display the head of the data.

In [50]:
data.head()

,Gender,Married,Education,TotalIncome,LoanAmount,CreditHistory,LoanStatus
0,Male,Yes,Graduate,6091.0,128.0,1.0,N
1,Male,Yes,Graduate,3000.0,66.0,1.0,Y
2,Male,Yes,Not Graduate,4941.0,120.0,1.0,Y
3,Male,No,Graduate,6000.0,141.0,1.0,Y
4,Male,Yes,Graduate,9613.0,267.0,1.0,Y


Data's columns:
* **Gender:** Applicant gender (Male/ Female)
* **Married:** Is the Applicant married? (Y/N)
* **Education:** Applicant Education (Graduate/ Not Graduate)
* **TotalIncome:** Applicant total income (sum of `ApplicantIncome` and `CoapplicantIncome` columns in the original dataset)
* **LoanAmount:** Loan amount in thousands
* **CreditHistory:** Credit history meets guidelines
* **LoanStatus** (Target)**:** Loan approved (Y/N)

### Data preprocessing

Define a list of categorical columns to encode.

In [51]:
categorical_columns = ["Gender", "Married", "Education", "LoanStatus"]

Encode categorical columns using the [`OrdinalEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html) of scikit learn.

In [52]:
data[categorical_columns] = OrdinalEncoder().fit_transform(data[categorical_columns])

Split into `X` and `y`.

In [53]:
X = data.drop(columns="LoanStatus")
y = data.LoanStatus

Normalize data using the [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) of scikit learn.

In [54]:
X[X.columns] = StandardScaler().fit_transform(X[X.columns])

Convert `y` type to `int` 

In [55]:
y = y.astype(int)

Split dataset into train and test sets.

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

## b. Dummy classifier

Build a dummy classifier that takes decisions randomly.

In [63]:
class DummyClassifier():
    
    def __init__(self, seed=None):
        """
        Initialize the class.
        """
        self.rng = np.random.default_rng(seed)
        self.classes_ = None
    
    def fit(self, X, y):
        """
        Fit the dummy classifier.
        
        Parameters
        ----------
        X : Numpy array or Pandas DataFrame of shape (n_samples, n_features)
            Training data.
        y : Numpy array or Pandas DataFrame of shape (n_samples,)
            Target values.
        """
        y = np.array(y)
        self.classes_ = np.unique(y)
        return self
    
    def predict(self, X):
        """
        Predict the class labels for the provided data.

        Parameters
        ----------
        X : Numpy array or Pandas DataFrame of shape (n_queries, n_features)
            Test samples.

        Returns
        -------
        y : Numpy array or Pandas DataFrame of shape (n_queries,)
            Class labels for each data sample.
        """
        if self.classes_ is None:
            raise RuntimeError("Must fit before calling predict.")
        n_queries = len(X)
        return self.rng.choice(self.classes_, size=n_queries)

Implement a function to evaluate the performance of a classification by computing the accuracy ($N_{correct}/N$).

In [64]:
def accuracy_score(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if(len(y_true) != len(y_pred)):
        raise ValueError("y_true and y_pred must have same length.")
    n_correct = np.sum(y_true == y_pred)
    return n_correct / len(y_true)

Compute the performance of the dummy classifier using the provided test set.

In [65]:
dummy = DummyClassifier(seed=42)
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Accuracy: %.2f%%" % (acc))

Accuracy: 0.58%


As the classifier is random, we can take the average across several runs using different seeds

In [66]:
accs = []

for seed in range (100):
    dummy = DummyClassifier(seed=seed)
    dummy.fit(X_train, y_train)
    y_pred = dummy.predict(X_test)
    accs.append(accuracy_score(y_test, y_pred))

mean = np.mean(accs)
std = np.std(accs)

print("Mean accuracy on 100 runs: %.2f%%" % (mean))
print("Standard deviation on 100 runs: %.2f%%" % (std))

Mean accuracy on 100 runs: 0.50%
Standard deviation on 100 runs: 0.05%


## c. K-Nearest Neighbors classifier

Build a K-Nearest Neighbors classifier using an Euclidian distance computation and a simple majority voting criterion.

In [73]:
class KNNClassifier():
    
    def __init__(self, n_neighbors=3):
        """
        Initialize the class.
        
        Parameters
        ----------
        n_neighbors : int, default=3
            Number of neighbors to use by default.
        """
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        """
        Fit the k-nearest neighbors classifier.
        
        Parameters
        ----------
        X : Numpy array or Pandas DataFrame of shape (n_samples, n_features)
            Training data.
        y : Numpy array or Pandas DataFrame of shape (n_samples,)
            Target values.
        """
        self.X_train = np.array(X, dtype=float)
        self.y_train = np.array(y)
        return self
    
    @staticmethod
    def _euclidian_distance(a, b):
        """
        Utility function to compute the euclidian distance.
        
        Parameters
        ----------
        a : Numpy array or Pandas DataFrame
            First operand.
        b : Numpy array or Pandas DataFrame
            Second operand.
        """
        a = np.array(a, dtype=float)
        b = np.array(b, dtype=float)
        return np.sqrt(np.sum((a - b)**2, axis=-1))
    
    def predict(self, X):
        """
        Predict the class labels for the provided data.

        Parameters
        ----------
        X : Numpy array or Pandas DataFrame of shape (n_queries, n_features)
            Test samples.

        Returns
        -------
        y : Numpy array or Pandas DataFrame of shape (n_queries,)
            Class labels for each data sample.
        """
        if self.X_train is None:
            raise RuntimeError("Must fit before calling predict.")
        X = np.array(X, dtype=float)
        predictions=[]

        for x in X:
            #distance btw X and all train points
            distances = self._euclidian_distance(self.X_train, x)

            #indices of the closest k points
            indices_neighbors = np.argsort(distances)[:self.n_neighbors]

            #class of the k neighbors
            class_neighbors = self.y_train[indices_neighbors]

            #maj vote
            classes, votes = np.unique(class_neighbors, return_counts=True)
            winning_class = classes[np.argmax(votes)]

            predictions.append(winning_class)
        return np.array(predictions)

Compute the performance of the system as a function of $k = 1...7$.

In [84]:
for k in range (1,8):
    knn = KNNClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred=knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print("k=", k, "Accuracy: ", (acc))

k= 1 Accuracy:  0.6979166666666666
k= 2 Accuracy:  0.6354166666666666
k= 3 Accuracy:  0.7916666666666666
k= 4 Accuracy:  0.7395833333333334
k= 5 Accuracy:  0.8125
k= 6 Accuracy:  0.78125
k= 7 Accuracy:  0.8020833333333334


Comments:

THe best performance is obtained with k = 5, with an accuracy of 81,3%. K=7 (80,2%) and k=3 (79,2%) are very close.

Using k=1, with a single neighbor, the decision depends entirely on the closest training point. This makes the classifier very sensitive to noise and outliners. Increasing k smoths the decision.

Odd values of k perform better than even values. This comes from ties in the vote. With an even k, neighbors can be split equally btw the 2 classes. In that case, our implementation always chooses the first class returned by np.unique, which is an arbitrary decision.

Compare to the dummy classifier (~50%), the KNN clearly learns something useful from data.


Run the KNN algorithm using only the features `TotalIncome` and `CreditHistory`.

In [85]:
def eval_knn(features, k=3):
    knn = KNNClassifier(n_neighbors=k)
    knn.fit(X_train[features], y_train)
    y_pred=knn.predict(X_test[features])
    acc = accuracy_score(y_test, y_pred)
    return acc

In [86]:
acc_c = eval_knn(["TotalIncome","CreditHistory"])
print("TotalIncome + CreditHistory :", round(acc_c, 3))

TotalIncome + CreditHistory : 0.781


Re-run the KNN algorithm using the features `TotalIncome`, `CreditHistory` and `Married`.

In [87]:
acc_c = eval_knn(["TotalIncome","CreditHistory", "Married"])
print("TotalIncome + CreditHistory + Married :", round(acc_c, 3))

TotalIncome + CreditHistory + Married : 0.865


Re-run the KNN algorithm using all features.

In [78]:
acc_c = eval_knn(list(X_train.columns))
print("All features :", round(acc_c, 3))

TotalIncome + CreditHistory : 0.792


The best performance is obtained with 3 features (86,5%), which is much better thant with all features (79,2%). Using more features does not necessarily improve the performace of KNN. Addin Married improves the accuracy by about 8 pts, but adding all the other features brings it back down to about the same level as with only 2 features.
KNN gives the same weihgt to all features in the Euclidean distance, so uninformative features add noise and make the nearest neighbors less relevant. This is also due to the curse of dimensionality. When the nbr of features increases, the training points become sparse and the nearest neighbors are no longer really close. The number of samples needed grows very quickly with the number of feaures, so with a small dataset, a few relevant features work better than many.